In [1]:
import pandas as pd
from pathlib import Path
import re

In [2]:
BASE_DIR = Path("..")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

lyrics_raw = pd.read_csv(RAW_DIR / "genius_lyrics_raw.csv")
tracks = pd.read_csv(PROCESSED_DIR / "spotify_tracks_clean.csv")

lyrics_raw.shape

(441, 11)

In [3]:
lyrics_raw["search_status"].value_counts()

search_status
exact            381
rejected          27
manual_review     12
not_found         11
instrumental      10
Name: count, dtype: int64

In [4]:
lyrics_raw.head()

,track_id,track_name,album_id,album_name,genius_title,genius_artist,genius_url,lyrics,title_score,artist_ok,search_status
0,6nt3AoYjkaqXMZhypTBky1,SWIM,6iPjmGZeonxBZ9r7Cjkezq,KEEP SWIMMING,SWIM,BTS,https://genius.com/Bts-swim-lyrics,"Swim, swim\nWater falling off your skin\nSwim,...",1.0,True,exact
1,7EytKcb3klVPpN5IW1sj1Y,SWIM with RM (Chill Hip Hop Remix),6iPjmGZeonxBZ9r7Cjkezq,KEEP SWIMMING,SWIM with RM (Chill Hip Hop Remix),BTS,https://genius.com/Bts-swim-with-rm-chill-hip-...,"Swim, swim\nWater fallin' off your skin\nSwim,...",1.0,True,exact
2,5dZLsPskKzph16LWo31uxL,SWIM with Jin (Alternative Rock Remix),6iPjmGZeonxBZ9r7Cjkezq,KEEP SWIMMING,SWIM with Jin (Alternative Rock Remix),BTS,https://genius.com/Bts-swim-with-jin-alternati...,"Swim, swim\nWater falling off your skin\nSwim,...",1.0,True,exact
3,5AL5OrvyIMPqKjl9iw3xO5,SWIM with SUGA (Melodic Techno Remix),6iPjmGZeonxBZ9r7Cjkezq,KEEP SWIMMING,SWIM with SUGA (Melodic Techno Remix),BTS,https://genius.com/Bts-swim-with-suga-melodic-...,"Swim, swim\nWater falling off your skin\nSwim,...",1.0,True,exact
4,3MCJY7lXCHa0UNIjsAucaJ,SWIM with j-hope (Afrobeat Remix),6iPjmGZeonxBZ9r7Cjkezq,KEEP SWIMMING,SWIM with j-hope (Afrobeat Remix),BTS,https://genius.com/Bts-swim-with-j-hope-afrobe...,"Swim, swim\nWater falling off your skin\nSwim,...",1.0,True,exact


In [5]:
lyrics_clean = lyrics_raw[
    lyrics_raw["search_status"] == "exact"
].copy()

lyrics_clean.shape

(381, 11)

In [6]:
def clean_genius_lyrics(text):
    if pd.isna(text):
        return None
    
    text = str(text)
    
    # Remove common Genius embed/footer noise
    text = re.sub(r"\d*Embed$", "", text)
    text = re.sub(r"You might also like", "", text)
    
    # Remove excessive whitespace
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    
    return text.strip()

In [9]:
lyrics_clean["lyrics_clean"] = lyrics_clean["lyrics"].apply(clean_genius_lyrics)

lyrics_clean[["track_name", "lyrics_clean"]].head()

,track_name,lyrics_clean
0,SWIM,"Swim, swim\nWater falling off your skin\nSwim,..."
1,SWIM with RM (Chill Hip Hop Remix),"Swim, swim\nWater fallin' off your skin\nSwim,..."
2,SWIM with Jin (Alternative Rock Remix),"Swim, swim\nWater falling off your skin\nSwim,..."
3,SWIM with SUGA (Melodic Techno Remix),"Swim, swim\nWater falling off your skin\nSwim,..."
4,SWIM with j-hope (Afrobeat Remix),"Swim, swim\nWater falling off your skin\nSwim,..."


In [10]:
lyrics_clean["lyrics_clean"].isna().sum()

np.int64(0)

In [11]:
lyrics_clean["lyrics_clean"].str.len().describe()

count     381.000000
mean     1491.406824
std       573.224425
min         8.000000
25%      1118.000000
50%      1504.000000
75%      1841.000000
max      3169.000000
Name: lyrics_clean, dtype: float64

## Check for duplicate lyrics


In [12]:
lyrics_clean["lyrics_clean"].duplicated().sum()

np.int64(115)

In [13]:
duplicates = lyrics_clean[
    lyrics_clean["lyrics_clean"].duplicated(keep=False)
].sort_values("lyrics_clean")

duplicates[[
    "track_name",
    "album_name", 
    "genius_title"
]]

,track_name,album_name,genius_title
386,Dynamite (Slow Jam Remix),Dynamite (NightTime Version),Dynamite (Slow Jam Remix)
392,Dynamite (Acoustic Remix),Dynamite (DayTime Version),Dynamite (Acoustic Remix)
390,Dynamite,Dynamite (DayTime Version),Dynamite
384,Dynamite,Dynamite (NightTime Version),Dynamite
60,Dynamite,Proof,Dynamite
...,...,...,...
312,Intro: Skool Luv Affair,Skool Luv Affair,Intro: Skool Luv Affair
302,Intro: Skool Luv Affair,Skool Luv Affair (Special Addition),Intro: Skool Luv Affair
131,DNA,Love Yourself 結 'Answer',DNA
55,DNA,Proof,DNA


## Empty or suspicious lyrics

In [14]:
lyrics_clean[
    lyrics_clean["lyrics_clean"].str.len() < 100
][["track_name", "lyrics_clean"]]

,track_name,lyrics_clean
273,Intro.,Wake up!
293,Interlude: What Are You Doing Now,뭐해?\n뭐해?\n뭐해?


## Check for Genius artifacts

Sometimes Genius leaves things like:

- Embed
- 123Embed
- Contributors
- Lyrics

In [16]:
patterns = [
    "embed", 
    "Contributors",
    "You might also like"
]

for p in patterns: 
    print(p, lyrics_clean["lyrics_clean"].str.contains(p, case=False).sum())

embed 0
Contributors 0
You might also like 0


In [17]:
lyrics_clean.sample(5)[
    ["track_name","lyrics_clean"]
]

,track_name,lyrics_clean
206,Begin,아무것도 없던 열다섯의 나\n세상은 참 컸어 너무 작은 나\n이제 난 상상할 수도 ...
207,Lie,내게 말해\n너의 달콤한 미소로 내게\n내게 말해\n속삭이듯 내 귓가에 말해\nDo...
223,Good Day,大丈夫だって (だって) Oh yeah\nいつかは Good day (Good day)...
97,ON - Japanese ver.,I can't understand what people are sayin'\nどのリ...
217,2! 3!,Been trying to tell you this\nI was supposed t...


## Lyrics statistics

In [18]:
lyrics_clean["character_count"] = lyrics_clean["lyrics_clean"].str.len()

lyrics_clean["word_count"] = (
    lyrics_clean["lyrics_clean"]
    .str.split()
    .str.len()
)

In [19]:
lyrics_clean[
    ["character_count", "word_count"]
].describe()

,character_count,word_count
count,381.000000,381.000000
mean,1491.406824,328.992126
std,573.224425,127.848461
min,8.000000,2.000000
25%,1118.000000,242.000000
50%,1504.000000,344.000000
75%,1841.000000,411.000000
max,3169.000000,778.000000


In [20]:
lyrics_clean.to_csv(
    PROCESSED_DIR / "song_lyrics_clean.csv",
    index=False
)

print("Saved:", PROCESSED_DIR / "song_lyrics_clean.csv")
print("Shape:", lyrics_clean.shape)

Saved: ../data/processed/song_lyrics_clean.csv
Shape: (381, 14)


In [21]:
lyrics_check = pd.read_csv(
    PROCESSED_DIR / "song_lyrics_clean.csv"
)

print(lyrics_check.shape)
lyrics_check.head()

(381, 14)


,track_id,track_name,album_id,album_name,genius_title,genius_artist,genius_url,lyrics,title_score,artist_ok,search_status,lyrics_clean,character_count,word_count
0,6nt3AoYjkaqXMZhypTBky1,SWIM,6iPjmGZeonxBZ9r7Cjkezq,KEEP SWIMMING,SWIM,BTS,https://genius.com/Bts-swim-lyrics,"Swim, swim\nWater falling off your skin\nSwim,...",1.0,True,exact,"Swim, swim\nWater falling off your skin\nSwim,...",1872,370
1,7EytKcb3klVPpN5IW1sj1Y,SWIM with RM (Chill Hip Hop Remix),6iPjmGZeonxBZ9r7Cjkezq,KEEP SWIMMING,SWIM with RM (Chill Hip Hop Remix),BTS,https://genius.com/Bts-swim-with-rm-chill-hip-...,"Swim, swim\nWater fallin' off your skin\nSwim,...",1.0,True,exact,"Swim, swim\nWater fallin' off your skin\nSwim,...",1816,362
2,5dZLsPskKzph16LWo31uxL,SWIM with Jin (Alternative Rock Remix),6iPjmGZeonxBZ9r7Cjkezq,KEEP SWIMMING,SWIM with Jin (Alternative Rock Remix),BTS,https://genius.com/Bts-swim-with-jin-alternati...,"Swim, swim\nWater falling off your skin\nSwim,...",1.0,True,exact,"Swim, swim\nWater falling off your skin\nSwim,...",1841,364
3,5AL5OrvyIMPqKjl9iw3xO5,SWIM with SUGA (Melodic Techno Remix),6iPjmGZeonxBZ9r7Cjkezq,KEEP SWIMMING,SWIM with SUGA (Melodic Techno Remix),BTS,https://genius.com/Bts-swim-with-suga-melodic-...,"Swim, swim\nWater falling off your skin\nSwim,...",1.0,True,exact,"Swim, swim\nWater falling off your skin\nSwim,...",2312,455
4,3MCJY7lXCHa0UNIjsAucaJ,SWIM with j-hope (Afrobeat Remix),6iPjmGZeonxBZ9r7Cjkezq,KEEP SWIMMING,SWIM with j-hope (Afrobeat Remix),BTS,https://genius.com/Bts-swim-with-j-hope-afrobe...,"Swim, swim\nWater falling off your skin\nSwim,...",1.0,True,exact,"Swim, swim\nWater falling off your skin\nSwim,...",1877,370
